In [ ]:

import sys
sys.path.insert(0, '..')

import json
import numpy as np
from pathlib import Path
from collections import defaultdict

# ─── Load results ───
results_path = Path('../data/eval_results.json')
if not results_path.exists():
    print(' No eval results. Run: python scripts/run_evaluation.py')
    raise SystemExit

with open(results_path) as f:
    raw = json.load(f)

if isinstance(raw, dict) and 'details' in raw:
    details = raw['details']
elif isinstance(raw, list):
    details = raw
else:
    details = []

valid = [r for r in details if 'error' not in r]
errors = [r for r in details if 'error' in r]
print(f'Loaded {len(details)} results ({len(valid)} valid, {len(errors)} errors)\n')

scores = np.array([r['score'] for r in valid])
hall_rates = np.array([r.get('hallucination_rate', 0) for r in valid])
iterations = np.array([r.get('iterations', 0) for r in valid])

# ─── Overall metrics ───
print('╔════════════════════════════════════════════╗')
print('║          EVALUATION RESULTS                ║')
print('╠════════════════════════════════════════════╣')
print(f'║  Questions:            {len(valid):>5}               ║')
print(f'║  Avg Confidence:       {scores.mean():>5.3f}               ║')
print(f'║  Std Dev:              {scores.std():>5.3f}               ║')
print(f'║  Hallucination Rate:   {hall_rates.mean():>5.1%}               ║')
print(f'║  Pass Rate (≥0.65):    {(scores >= 0.65).mean():>5.0%}               ║')
print(f'║  Avg Iterations:       {iterations.mean():>5.1f}               ║')
print('╚════════════════════════════════════════════╝')

# ─── Score distribution ───
print('\n\nScore Distribution:')
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.01]
hist, _ = np.histogram(scores, bins=bins)
mx = max(hist) if max(hist) > 0 else 1
for i in range(len(hist)):
    bar = '█' * int(hist[i] / mx * 25)
    print(f'  {bins[i]:.1f}-{bins[i+1]:.1f}: {hist[i]:>2} {bar}')

# ─── By question type ───
by_type = defaultdict(list)
for r in valid:
    by_type[r.get('type', 'unknown')].append(r)

print(f'\n\nBy Question Type:')
print(f'{"Type":<15} {"N":<5} {"Avg Score":<12} {"Hall Rate":<12} {"Pass%":<8}')
print('─' * 52)
for t in sorted(by_type):
    items = by_type[t]
    s = [r['score'] for r in items]
    h = [r.get('hallucination_rate', 0) for r in items]
    p = sum(1 for x in s if x >= 0.65) / len(s)
    print(f'{t:<15} {len(items):<5} {np.mean(s):<12.3f} {np.mean(h):<12.1%} {p:<8.0%}')

# ─── Best & worst ───
sorted_results = sorted(valid, key=lambda r: r['score'], reverse=True)

print(f'\n\n Top 3:')
for r in sorted_results[:3]:
    print(f'  {r["score"]:.2f} | {r.get("type",""):<10} | {r["question"][:50]}')

print(f'\n❌ Bottom 3:')
for r in sorted_results[-3:]:
    print(f'  {r["score"]:.2f} | {r.get("type",""):<10} | {r["question"][:50]}')

# ─── Iterations vs score ───
iter_groups = defaultdict(list)
for r in valid:
    iter_groups[r.get('iterations', 0)].append(r['score'])

print(f'\n\nIterations vs Score:')
for it in sorted(iter_groups):
    g = iter_groups[it]
    print(f'  {it} iter: avg={np.mean(g):.3f} (n={len(g)})')

# ─── Hallucination breakdown ───
hall_items = [r for r in valid if r.get('hallucination_rate', 0) > 0]
print(f'\n\nHallucination Breakdown:')
print(f'  Clean: {len(valid) - len(hall_items)}/{len(valid)} ({(len(valid)-len(hall_items))/len(valid):.0%})')
print(f'  With hallucination: {len(hall_items)}/{len(valid)} ({len(hall_items)/len(valid):.0%})')
if hall_items:
    for r in hall_items:
        print(f'    [{r.get("hallucination_rate",0):.0%}] {r["question"][:55]}')

# ─── README table ───
print(f'\n\n{"═" * 60}')
print(' For README.md:')
print('═' * 60)
print()
print('| Metric | Value |')
print('|--------|-------|')
print(f'| Avg Confidence | {scores.mean():.2f} |')
print(f'| Hallucination Rate | {hall_rates.mean():.0%} |')
print(f'| Pass Rate (≥0.65) | {(scores >= 0.65).mean():.0%} |')
print(f'| Avg Iterations | {iterations.mean():.1f} |')
print(f'| Questions | {len(valid)} |')

# ─── Recommendations ───
print(f'\n\n📌 Recommendations:')
if scores.mean() < 0.7:
    print('  • Score < 0.7: Improve retrieval quality or add more docs')
if hall_rates.mean() > 0.15:
    print('  • Hallucination > 15%: Tighten grounding constraints')
if iterations.mean() > 2.0:
    print('  • High iterations: Retrieval may be returning weak results')

print('\n Evaluation analysis complete')